In [1]:
## Exercise: which of the extra augmentations actually help?
##
## Continues sessions/augmentation.ipynb, and uses build_model, X_train, Y_train,
## X_validation, Y_validation, keras, np, pickle, time and SEED from it.
##
## One layer is added at a time on top of flip + translation, so that a result can
## be attributed to a particular transformation rather than to "more augmentation".
## The two controls and the all-four arm are trained on the same 20-epoch budget as
## the rest, so that every arm has a checkpoint and can be measured the same way.
##
## The three traps from the session, all applied below:
##   - RandomContrast and RandomErasing take value_range=(0, 1), because this
##     notebook normalized the pixels to [0, 1]. With the default (0, 255) the
##     effect is computed on a scale 255 times too large and nothing raises.
##   - CIFAR objects are upright, so the rotation stays small: factor=0.05 is
##     about +/- 18 degrees.
##   - RandomZoom's factors are fractions, and the fill matters for the same
##     reason it does for RandomTranslation.

EPOCHS = 20

def base():
    """Fresh layer instances each call - a layer belongs to the model it is built into."""
    return [
        keras.layers.RandomFlip('horizontal'),
        keras.layers.RandomTranslation(0.125, 0.125, fill_mode='constant', fill_value=0.0),
    ]

EXTRA = {
    'rotation': lambda: keras.layers.RandomRotation(0.05, fill_mode='constant', fill_value=0.0),
    'zoom':     lambda: keras.layers.RandomZoom(0.1, 0.1, fill_mode='constant', fill_value=0.0),
    'contrast': lambda: keras.layers.RandomContrast(0.2, value_range=(0, 1)),
    'erasing':  lambda: keras.layers.RandomErasing(0.25, value_range=(0, 1)),
}

PROBE_EXTRA = {
    'rotation_reflect': lambda: keras.layers.RandomRotation(0.05, fill_mode='reflect'),
    'zoom_reflect':     lambda: keras.layers.RandomZoom(0.1, 0.1, fill_mode='reflect'),
}


def augment_for(arm):
    """The augmentation block for one arm, or None for the unaugmented control."""
    if arm == 'noaug':
        return None
    if arm == 'simple':
        extra = []
    elif arm == 'all':
        extra = [make() for make in EXTRA.values()]
    elif arm in PROBE_EXTRA:
        extra = [PROBE_EXTRA[arm]()]
    else:
        extra = [EXTRA[arm]()]
    return keras.Sequential(base() + extra, name='augment_' + arm)

## The two probe arms below are not part of the exercise as set: they repeat
## rotation and zoom with reflected fill instead of black borders, to test an
## explanation for the result. See the discussion at the end.
PROBES = [
    ('+ rotation (reflect)', 'rotation_reflect'),
    ('+ zoom (reflect)',     'zoom_reflect'),
]

ARMS = [
    ('no augmentation',    'noaug'),
    ('flip + translation', 'simple'),
    ('+ rotation',         'rotation'),
    ('+ zoom',             'zoom'),
    ('+ contrast',         'contrast'),
    ('+ erasing',          'erasing'),
    ('all four',           'all'),
]

Training all of them, which is what produced the checkpoints loaded below:

In [2]:
## About 45 minutes per arm, so roughly five hours for all seven: not a class
## exercise. Run one or two arms yourself and load the rest.
##
# for _, arm in ARMS + PROBES:
#     model = build_model(augment=augment_for(arm))
#     model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
#     t0 = time.time()
#     history = model.fit(
#         x=X_train, y=Y_train, batch_size=64, epochs=EPOCHS,
#         validation_data=(X_validation, Y_validation), verbose=2,
#     ).history
#     model.save('../data/cifar10_aug_{}.keras'.format(arm))
#     with open('../data/cifar10_aug_{}_history.p'.format(arm), 'wb') as f:
#         pickle.dump(history, f)
#     print('{}: {:.0f}s'.format(arm, time.time() - t0))

Every arm is measured the way the session measured its two: on the **clean** training
images in inference mode, so that dropout is off and the augmentation layers are inert.
The `train` column below is therefore comparable across arms, which the `accuracy`
recorded during training is not.

In [3]:
truth_train = Y_train.argmax(axis=1)
truth_validation = Y_validation.argmax(axis=1)


_measured = {}


def measure(arm):
    """Clean-data accuracy in inference mode, plus the curve statistics. Cached."""
    if arm in _measured:
        return _measured[arm]
    model = keras.models.load_model('../data/cifar10_aug_{}.keras'.format(arm))
    with open('../data/cifar10_aug_{}_history.p'.format(arm), 'rb') as f:
        history = pickle.load(f)
    train = (model.predict(X_train, batch_size=512, verbose=0).argmax(axis=1)
             == truth_train).mean()
    validation = (model.predict(X_validation, batch_size=512, verbose=0).argmax(axis=1)
                  == truth_validation).mean()
    curve = history['val_accuracy']
    best = int(np.argmax(curve))
    _measured[arm] = (train, validation, curve[best], best + 1, np.std(curve[-5:]))
    return _measured[arm]


def report(rows, baseline=None):
    print('{:<26s}{:>9s}{:>12s}{:>8s}{:>11s}{:>7s}{:>7s}{:>9s}'.format(
        'on clean data', 'train', 'validation', 'gap', 'best val', 'epoch', 'sd5', 'vs ctrl'))
    for label, arm in rows:
        train, validation, best, best_epoch, sd5 = measure(arm)
        delta = '' if baseline is None else '{:+8.2f}'.format(100 * (validation - baseline))
        print('{:<26s}{:>9.2%}{:>12.2%}{:>8.2%}{:>11.2%}{:>7d}{:>7.2f}{:>9s}'.format(
            label, train, validation, train - validation, best, best_epoch, 100 * sd5, delta))


control = measure('simple')[1]
report(ARMS, baseline=control)

singles = sum(measure(a)[1] - control for a in ('rotation', 'zoom', 'contrast', 'erasing'))
print('\nthe four single-layer deficits sum to {:+.2f} pp; all four together is {:+.2f} pp'.format(
    100 * singles, 100 * (measure('all')[1] - control)))

on clean data                 train  validation     gap   best val  epoch    sd5  vs ctrl


no augmentation              96.96%      76.71%  20.25%     77.07%     10   0.25    -0.71
flip + translation           80.79%      77.42%   3.37%     77.71%     17   1.03    +0.00


+ rotation                   74.94%      72.06%   2.88%     74.64%     17   1.07    -5.36


+ zoom                       78.29%      75.07%   3.22%     75.55%     17   1.34    -2.35


+ contrast                   80.92%      76.74%   4.18%     78.38%     17   1.20    -0.68


+ erasing                    80.62%      77.39%   3.23%     77.39%     20   1.21    -0.03


all four                     70.88%      68.24%   2.64%     71.60%     15   1.95    -9.18

the four single-layer deficits sum to -8.42 pp; all four together is -9.18 pp


### Does the black border explain it?

`RandomRotation` and `RandomZoom` with `fill_mode='constant'` put black wedges and borders
on the training images. Validation images never have them, so a natural guess is that the
two arms suffer because their training data no longer looks like their validation data.

That is a testable guess: repeat the two arms with `fill_mode='reflect'`, which fills the
vacated region with mirrored image content instead, and see how much of the deficit comes
back.

In [4]:
report([('flip + translation', 'simple')] + PROBES, baseline=control)

print()
for solid, reflected in (('rotation', 'rotation_reflect'), ('zoom', 'zoom_reflect')):
    constant = 100 * (measure(solid)[1] - control)
    reflect = 100 * (measure(reflected)[1] - control)
    print('{:<10s} constant {:+.2f} pp  ->  reflect {:+.2f} pp   recovered {:.2f} of {:.2f}'.format(
        solid, constant, reflect, reflect - constant, abs(constant)))

on clean data                 train  validation     gap   best val  epoch    sd5  vs ctrl
flip + translation           80.79%      77.42%   3.37%     77.71%     17   1.03    +0.00


+ rotation (reflect)         76.16%      73.36%   2.80%     74.63%     16   1.12    -4.06


+ zoom (reflect)             78.50%      75.35%   3.15%     75.55%     17   1.53    -2.07

rotation   constant -5.36 pp  ->  reflect -4.06 pp   recovered 1.30 of 5.36
zoom       constant -2.35 pp  ->  reflect -2.07 pp   recovered 0.28 of 2.35


## What the ablation shows

**The answer to the exercise is that the extra augmentations do not help, and two of the four do real damage.** Against the `flip + translation` control at epoch 20, on clean data:

| arm | vs control |
|---|---|
| + contrast | −0.68 pp |
| + erasing | −0.03 pp |
| + zoom | −2.35 pp |
| + rotation | −5.36 pp |
| all four | −9.18 pp |

The session measured about a point of run-to-run slack on identical code, so **contrast and erasing are indistinguishable from free**, and rotation and zoom are unambiguously costly. The four deficits sum to −8.42 against −9.18 for all four together, so they stack roughly additively.

**This is not "too much augmentation".** That was our first guess and the numbers do not support it. Contrast and erasing add regularization and cost nothing; if the problem were simply regularization strength, they would hurt too. The damage is specific to the two transformations that move pixels geometrically.

**It is not over-regularization at all, on the evidence here.** Look at the `gap` column rather than the accuracies. The `+ rotation` arm has a *smaller* gap than the control — 2.88 against 3.37 — while its validation accuracy is 5.4 points worse. A regularizer that was working too hard would show a small gap and a *high* validation accuracy; a small gap with a low one means the model is not fitting the task, not that it is being held back from overfitting it. Every heavier arm has lower **clean-train** accuracy than the control: 74.94% for rotation against 80.79%. They are fitting less, not generalizing less.

### A hypothesis we tested and rejected

The obvious explanation is the black border: with `fill_mode='constant'` every rotated or zoomed training image acquires wedges of black that no validation image has, so the two distributions drift apart. It predicts that reflected fill should recover the loss.

It does not:

| arm | constant fill | reflect fill | recovered |
|---|---|---|---|
| + rotation | −5.36 pp | −4.06 pp | 1.30 of 5.36 |
| + zoom | −2.35 pp | −2.07 pp | 0.28 of 2.35 |

Reflected fill recovers about a quarter of rotation's deficit and, for zoom, an amount inside the run-to-run noise. **The fill is a small part of the story at most.** Most of the cost is the geometric transformation itself, not the border it leaves behind. This is worth doing in front of students: it is a clean, cheap prediction, and it comes out mostly negative.

### What we cannot separate

The honest limit of this experiment. Every heavier arm is *less converged* at 20 epochs — lower training accuracy as well as lower validation accuracy — and harder augmentation slows fitting, which is the confound the session already flagged. So we cannot distinguish "rotation destroys label-relevant structure" from "rotation merely needs more epochs than we gave it". The evidence is consistent with both.

Two things sharpen that. First, at this 20-epoch budget the control itself beats *no augmentation at all* by only 0.71 points, which is inside the noise — the session needed 50 epochs before flip and translation showed their 5.8-point margin. **This whole exercise runs at a budget where augmentation has barely begun to pay off**, so read every row as "at 20 epochs", not as a verdict. Second, the settling test is cheap to specify: re-run the `+ rotation` arm at 50 epochs and see whether it closes on the control the way the control closed on no augmentation. We did not run it.

### The one thing that is not budget-dependent

`no augmentation` reaches **96.96%** on its own training set at epoch 20, against 80.79% for the control, and its gap is 20.25 points against 3.37. Whatever else is unresolved, the augmented models are not memorizing and the unaugmented one is — which is the session's point, visible at a fifth of its budget.

### Worth trying, not measured here

Drop `value_range=(0, 1)` from `RandomContrast` and `RandomErasing` and re-run. With the default `(0, 255)` both layers compute their effect on a scale 255 times too large for our normalized pixels. Nothing raises, the training loop runs exactly as before, and the numbers simply get worse. It is worth seeing how much worse, because that is what the whole class of silent augmentation bugs looks like from the outside.